# Notebook 02 - Eksplorasi Data Mentah LIN/PBN

Notebook ini menampilkan contoh board dan struktur record hasil parsing LIN/PBN sebelum DDS, pembuatan label, dan pembagian dataset. Notebook ini tidak bergantung pada file `data/processed/` yang baru dibuat di Notebook 03.

**Input:** `data/raw/` (LIN) dan `data/raw_pbn/` (PBN).
**Tahap berikutnya:** Notebook 03 membentuk label DDS, dua varian dataset, dan split deal-aware.


## 0. Setup


In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')

_root_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _root_candidates if (p / "configs" / "config.yaml").exists() and (p / "src").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Tidak menemukan root repo; jalankan notebook dari repo atau folder notebooks.")
sys.path.insert(0, str(ROOT))

print(f"Repo root: {ROOT}")
print("Notebook ini hanya membaca sampel sumber LIN/PBN.")

Repo root: d:\Bridge-Prediction
Notebook ini hanya membaca sampel sumber LIN/PBN.


## 1. Tabel Board Mentah Sebelum Feature Engineering

Satu baris = satu board hasil parsing. Tangan ditulis per suit (`♠ | ♥ | ♦ | ♣`), dan auction ditulis urut dari dealer. Sampel berasal dari beberapa file LIN dan PBN; dataset lengkap dibangun pada Notebook 03.


In [2]:
from src.parser import LINParser, PBNParser

lin_sample_files = sorted((ROOT / 'data' / 'raw').glob('*.lin'))[:3]
pbn_sample_files = sorted({p for p in (ROOT / 'data' / 'raw_pbn').iterdir()
                           if p.suffix.lower() == '.pbn'})[:2]

raw_boards = []
for f in lin_sample_files:
    raw_boards += LINParser().parse_file(f)
for f in pbn_sample_files:
    raw_boards += PBNParser().parse_file(f)

def hand_str(hand):
    if hand is None:
        return ''
    d = hand.as_dict()
    return ' | '.join(''.join(d.get(s, [])) or '-' for s in ('S', 'H', 'D', 'C'))

def contract_str(b):
    if not b.contract_level:
        return 'PASS'
    return f"{b.contract_level}{b.contract_strain}{'xx' if b.redoubled else 'x' if b.doubled else ''}"

raw_df = pd.DataFrame([{
    'file'     : Path(b.source_file).name,
    'room'     : b.room,
    'board'    : b.board_number,
    'dealer'   : b.dealer,
    'vul'      : b.vulnerability,
    'North'    : hand_str(b.hands.get('N')),
    'East'     : hand_str(b.hands.get('E')),
    'South'    : hand_str(b.hands.get('S')),
    'West'     : hand_str(b.hands.get('W')),
    'lelang'   : ' '.join(b.auction),
    'kontrak'  : contract_str(b),
    'declarer' : b.declarer,
    'hasil'    : b.result,
    'trik'     : b.tricks_made,
} for b in raw_boards])

print(f'{len(raw_boards)} board dari {len(lin_sample_files)} file LIN + {len(pbn_sample_files)} file PBN')
display(raw_df.head(15))

130 board dari 3 file LIN + 2 file PBN


,file,room,board,dealer,vul,North,East,South,West,lelang,kontrak,declarer,hasil,trik
0,85168.lin,open,13,N,Both,KT87 | T76 | 873 | 743,Q642 | J532 | 2 | AQ62,J9 | AQ9 | QJT96 | J95,A53 | K84 | AK54 | KT8,p p 1D 1N p 2C p 2D p 2N p 3N p p p,3N,W,-1.0,8.0
1,85168.lin,closed,13,N,Both,KT87 | T76 | 873 | 743,Q642 | J532 | 2 | AQ62,J9 | AQ9 | QJT96 | J95,A53 | K84 | AK54 | KT8,p p 1D 1N p 2C p 2D p 3N p p p,3N,W,NaN,NaN
2,85168.lin,open,14,E,None,QJ983 | KT2 | T75 | QT,K76 | 5 | QJ64 | AK752,AT542 | AJ6 | A983 | 4,- | Q98743 | K2 | J9863,1C 1S d 2N p 4S 5C p p d p p p,5Cx,E,0.0,11.0
3,85168.lin,closed,14,E,None,QJ983 | KT2 | T75 | QT,K76 | 5 | QJ64 | AK752,AT542 | AJ6 | A983 | 4,- | Q98743 | K2 | J9863,1C 1S d 2C p 4S 5C p p d p p p,5Cx,E,0.0,11.0
4,85168.lin,open,15,S,NS,AT873 | K42 | 93 | Q85,K4 | A983 | AJ87 | K63,652 | J65 | T62 | AJ94,QJ9 | QT7 | KQ54 | T72,p p p 1N p 3N p p p,3N,E,-1.0,8.0
5,85168.lin,closed,15,S,NS,AT873 | K42 | 93 | Q85,K4 | A983 | AJ87 | K63,652 | J65 | T62 | AJ94,QJ9 | QT7 | KQ54 | T72,p p p 1N p 3N p p p,3N,E,NaN,NaN
6,85168.lin,open,16,W,EW,KQ864 | 65 | 72 | J754,A97 | 72 | KJ953 | A63,J52 | AT843 | Q4 | QT9,T3 | KQJ9 | AT86 | K82,1D 1S 2S 3S p p 3N p p p,3N,E,NaN,NaN
7,85168.lin,closed,16,W,EW,KQ864 | 65 | 72 | J754,A97 | 72 | KJ953 | A63,J52 | AT843 | Q4 | QT9,T3 | KQJ9 | AT86 | K82,1D 2D d 3H d 3S 3N p p p,3N,E,0.0,9.0
8,85168.lin,open,17,N,None,J9832 | T852 | 974 | 8,74 | A73 | A853 | K764,Q6 | K964 | KQJ2 | J52,AKT5 | QJ | T6 | AQT93,p p 1D 2C p 2D p 2S p 3N p p p,3N,E,1.0,10.0
9,85168.lin,closed,17,N,None,J9832 | T852 | 974 | 8,74 | A73 | A853 | K764,Q6 | K964 | KQJ2 | J52,AKT5 | QJ | T6 | AQT93,p 1C 1D 1H p 1N p 3N p p p,3N,E,0.0,9.0


## 2. Tinjauan Record PBN

Tabel berikut memisahkan contoh yang berasal dari file PBN agar format sumber ini mudah diperiksa.

In [3]:
# Board yang sama dari sisi PBN
display(raw_df[raw_df['file'].str.lower().str.endswith('.pbn')].head(10))

,file,room,board,dealer,vul,North,East,South,West,lelang,kontrak,declarer,hasil,trik
66,angelfire_bermud00_52B1USNO.PBN,open,1|Brogeland|Saelensminde|Silverman|Wolfson,N,None,85 | AQ83 | 943 | 7542,KQT | 6 | 852 | KQJT96,J732 | JT972 | AJT7 | -,A964 | K54 | KQ6 | A83,Pass 1C X XX 1H 2C Pass 3NT Pass Pass 4H X Pas...,4Hx,N,-2.0,8.0
67,angelfire_bermud00_52B1USNO.PBN,closed,1|Austberg|Helgemo|Mahmood|Rosenberg,N,None,85 | AQ83 | 943 | 7542,KQT | 6 | 852 | KQJT96,J732 | JT972 | AJT7 | -,A964 | K54 | KQ6 | A83,Pass 1C Pass 1S Pass 2C Pass 2D Pass 2S Pass 3...,6C,E,-1.0,11.0
68,angelfire_bermud00_52B1USNO.PBN,open,2|Brogeland|Saelensminde|Silverman|Wolfson,E,NS,QJT | QJ5 | KT432 | 53,A983 | T93 | Q85 | A42,K74 | K8762 | 6 | KT86,652 | A4 | AJ97 | QJ97,Pass Pass 1C 1D 1S X XX Pass 1NT Pass Pass Pass,1N,E,-1.0,6.0
69,angelfire_bermud00_52B1USNO.PBN,closed,2|Austberg|Helgemo|Mahmood|Rosenberg,E,NS,QJT | QJ5 | KT432 | 53,A983 | T93 | Q85 | A42,K74 | K8762 | 6 | KT86,652 | A4 | AJ97 | QJ97,Pass Pass 1NT Pass Pass 2C X 2D 2NT Pass Pass ...,2N,W,-4.0,4.0
70,angelfire_bermud00_52B1USNO.PBN,open,3|Brogeland|Saelensminde|Silverman|Wolfson,S,EW,J64 | KT974 | 7 | KT92,A52 | 5 | J95432 | A43,T973 | 86 | KQT6 | J87,KQ8 | AQJ32 | A8 | Q65,Pass 1H Pass 1NT Pass 3NT Pass Pass Pass,3N,E,0.0,9.0
71,angelfire_bermud00_52B1USNO.PBN,closed,3|Austberg|Helgemo|Mahmood|Rosenberg,S,EW,J64 | KT974 | 7 | KT92,A52 | 5 | J95432 | A43,T973 | 86 | KQT6 | J87,KQ8 | AQJ32 | A8 | Q65,Pass 1H Pass 1NT Pass 2NT Pass 3NT Pass Pass Pass,3N,E,0.0,9.0
72,angelfire_bermud00_52B1USNO.PBN,open,4|Brogeland|Saelensminde|Silverman|Wolfson,W,Both,QJ9 | T4 | Q9432 | 864,764 | 8653 | J65 | A52,T | AKQJ92 | AKT7 | K9,AK8532 | 7 | 8 | QJT73,1S Pass Pass 2S 3C Pass 3S 4H Pass Pass 4S Pas...,5H,S,0.0,11.0
73,angelfire_bermud00_52B1USNO.PBN,closed,4|Austberg|Helgemo|Mahmood|Rosenberg,W,Both,QJ9 | T4 | Q9432 | 864,764 | 8653 | J65 | A52,T | AKQJ92 | AKT7 | K9,AK8532 | 7 | 8 | QJT73,1S Pass 1NT 4H 4S Pass Pass X Pass Pass Pass,4Sx,W,-1.0,9.0
74,angelfire_bermud00_52B1USNO.PBN,open,5|Brogeland|Saelensminde|Silverman|Wolfson,N,NS,KQ9 | KT74 | A9 | AT52,T7532 | - | J854 | KQJ7,J64 | AQ9 | KT62 | 984,A8 | J86532 | Q73 | 63,1NT Pass 2D Pass 2H Pass 3NT Pass Pass Pass,3N,N,1.0,10.0
75,angelfire_bermud00_52B1USNO.PBN,closed,5|Austberg|Helgemo|Mahmood|Rosenberg,N,NS,KQ9 | KT74 | A9 | AT52,T7532 | - | J854 | KQJ7,J64 | AQ9 | KT62 | 984,A8 | J86532 | Q73 | 63,1NT Pass 3NT Pass Pass Pass,3N,N,1.0,10.0


## Ringkasan dan Tahap Berikutnya

Notebook ini memeriksa bentuk record LIN/PBN sebelum feature engineering. Dataset penuh, label DDS, dua varian fitur, dan split deal-aware dibangun di Notebook 03.